In [ ]:
# Run this whole cell in Colab or Jupyter.

!pip -q install yfinance openpyxl pandas numpy

from pathlib import Path
import numpy as np
import pandas as pd
import yfinance as yf

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill
from openpyxl.comments import Comment

# -------------------------
# Configuration
# -------------------------

OUTPUT_PATH = Path("qaoa_rqp_tech_digital_infrastructure_40_blocks.xlsx")

TOTAL_PORTFOLIO_USD = 2_000_000
BLOCK_SIZE_USD = 100_000

N_FIXED_BLOCKS = 10
N_VARIABLE_BLOCKS = 30
N_SELECT_VARIABLE = 10

FIXED_BUDGET_USD = N_FIXED_BLOCKS * BLOCK_SIZE_USD
VARIABLE_BUDGET_USD = N_SELECT_VARIABLE * BLOCK_SIZE_USD

FIXED_ASSETS = [
    ("FIX_01", "MSFT", "Microsoft", "Software / Cloud"),
    ("FIX_02", "AAPL", "Apple", "Devices / Ecosystem"),
    ("FIX_03", "NVDA", "NVIDIA", "Semiconductors / AI"),
    ("FIX_04", "GOOGL", "Alphabet", "Digital Platforms / Cloud"),
    ("FIX_05", "AMZN", "Amazon", "Cloud / E-Commerce"),
    ("FIX_06", "META", "Meta Platforms", "Digital Platforms"),
    ("FIX_07", "AVGO", "Broadcom", "Semiconductors / Infrastructure"),
    ("FIX_08", "ADBE", "Adobe", "Software"),
    ("FIX_09", "CRM", "Salesforce", "Enterprise Software"),
    ("FIX_10", "ASML", "ASML", "Semiconductor Equipment"),
]

VARIABLE_ASSETS = [
    ("VAR_01", "AMD", "Advanced Micro Devices", "Semiconductors / AI"),
    ("VAR_02", "TSM", "TSMC", "Semiconductors / Foundry"),
    ("VAR_03", "QCOM", "Qualcomm", "Semiconductors / Mobile"),
    ("VAR_04", "INTC", "Intel", "Semiconductors / Foundry"),
    ("VAR_05", "MU", "Micron Technology", "Memory / Semiconductors"),

    ("VAR_06", "ORCL", "Oracle", "Enterprise Software / Cloud"),
    ("VAR_07", "NOW", "ServiceNow", "Enterprise Software"),
    ("VAR_08", "SNOW", "Snowflake", "Data / Cloud"),
    ("VAR_09", "DDOG", "Datadog", "Cloud Monitoring"),
    ("VAR_10", "MDB", "MongoDB", "Database Software"),

    ("VAR_11", "PANW", "Palo Alto Networks", "Cybersecurity"),
    ("VAR_12", "CRWD", "CrowdStrike", "Cybersecurity"),
    ("VAR_13", "ZS", "Zscaler", "Cybersecurity"),
    ("VAR_14", "FTNT", "Fortinet", "Cybersecurity"),
    ("VAR_15", "NET", "Cloudflare", "Edge / Cybersecurity"),

    ("VAR_16", "V", "Visa", "Payments Infrastructure"),
    ("VAR_17", "MA", "Mastercard", "Payments Infrastructure"),
    ("VAR_18", "PYPL", "PayPal", "Digital Payments"),
    ("VAR_19", "FI", "Fiserv", "Financial Technology"),
    ("VAR_20", "FIS", "FIS", "Financial Technology"),

    ("VAR_21", "EQIX", "Equinix", "Data Centers"),
    ("VAR_22", "DLR", "Digital Realty", "Data Centers"),
    ("VAR_23", "AMT", "American Tower", "Digital Infrastructure"),
    ("VAR_24", "CCI", "Crown Castle", "Digital Infrastructure"),
    ("VAR_25", "CSCO", "Cisco", "Networking Infrastructure"),

    ("VAR_26", "SHOP", "Shopify", "Digital Commerce"),
    ("VAR_27", "UBER", "Uber", "Digital Platforms"),
    ("VAR_28", "SPOT", "Spotify", "Digital Platforms"),
    ("VAR_29", "MELI", "MercadoLibre", "Digital Commerce"),
    ("VAR_30", "SE", "Sea Limited", "Digital Commerce / Gaming"),
]

ALL_ASSETS = FIXED_ASSETS + VARIABLE_ASSETS
TICKERS = [x[1] for x in ALL_ASSETS]
NAME_MAP = {ticker: company for _, ticker, company, _ in ALL_ASSETS}
SUBSECTOR_MAP = {ticker: subsector for _, ticker, _, subsector in ALL_ASSETS}

SETTINGS_ROWS = [
    ("demo_case_name", "USD 2m Technology & Digital Infrastructure Sleeve", "Professional demo case"),
    ("total_portfolio_usd", TOTAL_PORTFOLIO_USD, "Target total portfolio size after rebalancing"),
    ("fixed_budget_usd", FIXED_BUDGET_USD, "Capital already allocated to fixed existing positions"),
    ("budget_usd", VARIABLE_BUDGET_USD, "Budget available for newly selected variable positions"),
    ("block_size_usd", BLOCK_SIZE_USD, "Nominal size of each asset block"),
    ("fixed_blocks", N_FIXED_BLOCKS, "Number of fixed existing positions"),
    ("variable_blocks", N_VARIABLE_BLOCKS, "Number of variable candidate blocks"),
    ("target_new_blocks", N_SELECT_VARIABLE, "Target number of new variable blocks to select"),
    ("risk_free_rate_annual", 0.04, "Annual risk-free rate used in excess-return reward"),
    ("lambda_budget", 50.0, "Budget deviation penalty"),
    ("lambda_variance", 6.0, "Variance contribution weight"),
    ("lambda_exclusive", 0.0, "Unused in current notebook; kept for transparency"),

    ("top_n_export", 20, "Number of candidate portfolios exported to overview sheets"),
    ("refresh_market_data", 0, "1 = refresh prices/returns/volatility from yfinance before solving"),

    ("enable_qaoa", 1, "1 = run QAOA, 0 = skip QAOA"),
    ("qaoa_p", 1, "QAOA depth / number of layers"),
    ("qaoa_maxiter", 60, "Optimizer iteration budget for QAOA"),
    ("qaoa_shots", 4096, "Shot count used when exact probability extraction is disabled"),
    ("qaoa_exact_probability_max_qubits", 20, "Use exact qml.probs only if n is at or below this threshold"),
    ("qaoa_max_qubits_allowed", 32, "Auto-skip QAOA if number of variable decision blocks exceeds this threshold"),

    ("qaoa_export_mode", "top_k", "top_k or all_filtered"),
    ("qaoa_min_probability_to_export", 1e-12, "Exact-mode export ignores states below this probability"),
    ("qaoa_max_export_rows", 5000, "Maximum number of QAOA rows kept for export"),
    ("qaoa_export_feasible_only", 0, "1 = only export states within feasibility tolerance"),
    ("qaoa_feasibility_budget_tolerance_usd", 2500.0, "Abs budget gap tolerance used by feasible-only export"),
    ("qaoa_export_sort_by", "probability", "probability, qubo_value, sharpe_like, portfolio_return, abs_budget_gap"),

    ("enable_classical_search", 1, "1 = run classical local-search heuristic, 0 = skip it"),
    ("classical_max_qubits_allowed", 40, "Auto-skip classical heuristic if number of variable decision blocks exceeds this threshold"),
    ("classical_random_search_samples", 8000, "Number of random classical starts"),
    ("classical_local_search_starts", 40, "Number of single-hot classical starts"),
    ("classical_max_neighbor_evals", 200000, "Safety cap on total 1-flip neighbor evaluations in greedy search"),

    ("overview_classical_pool", 300, "How many classical candidates feed the combined overview pool"),
    ("overview_qaoa_pool", 500, "How many QAOA candidates feed the combined overview pool"),
    ("result_candidate_limit_per_solver", 500, "How many rows to write to each raw solver output sheet"),
    ("rng_seed", 42, "Random seed for classical search and QAOA initialization"),
]

# -------------------------
# Download market data
# -------------------------

prices = yf.download(
    tickers=TICKERS,
    period="12mo",
    interval="1d",
    auto_adjust=True,
    progress=False,
)["Close"]

if isinstance(prices, pd.Series):
    prices = prices.to_frame()

prices = prices.dropna(how="all").ffill().dropna()

missing_tickers = [ticker for ticker in TICKERS if ticker not in prices.columns]
if missing_tickers:
    raise ValueError(f"Missing yfinance data for: {missing_tickers}")

rets = prices.pct_change().dropna()

total_return_12m = prices.iloc[-1] / prices.iloc[0] - 1
ann_vol = rets.std() * np.sqrt(252)
mean_daily = rets.mean()
std_daily = rets.std()
daily_cov = rets.cov()
annual_cov = daily_cov * 252
latest_price = prices.iloc[-1]

# -------------------------
# Build asset block rows
# -------------------------

asset_rows = []

for block_id, ticker, company, subsector in FIXED_ASSETS:
    current_price = float(latest_price[ticker])
    shares = max(int(round(BLOCK_SIZE_USD / current_price)), 1)
    market_cost = shares * current_price

    asset_rows.append({
        "decision_id": block_id,
        "Decision Type": "fixed",
        "Ticker": ticker,
        "Company": company,
        "Theme": "Technology & Digital Infrastructure",
        "Subsector": subsector,
        "Asset Group": ticker,
        "Option Label": "fixed_100k",
        "Shares": shares,
        "Approx Cost USD": BLOCK_SIZE_USD,
        "Indicative Market Cost USD": market_cost,
        "Expected Return Proxy": float(total_return_12m[ticker]),
        "Annual Volatility": float(ann_vol[ticker]),
        "Current Price (USD)": current_price,
        "Mean Daily Return": float(mean_daily[ticker]),
        "Std Daily Return": float(std_daily[ticker]),
        "Allowed": 1,
        "Fixed Initial Holding": 1,
        "Variable Candidate": 0,
        "Price Source Status": "Imported via yfinance",
        "Source URL": "https://pypi.org/project/yfinance/",
    })

for block_id, ticker, company, subsector in VARIABLE_ASSETS:
    current_price = float(latest_price[ticker])
    shares = max(int(round(BLOCK_SIZE_USD / current_price)), 1)
    market_cost = shares * current_price

    asset_rows.append({
        "decision_id": block_id,
        "Decision Type": "variable",
        "Ticker": ticker,
        "Company": company,
        "Theme": "Technology & Digital Infrastructure",
        "Subsector": subsector,
        "Asset Group": ticker,
        "Option Label": "new_100k",
        "Shares": shares,
        "Approx Cost USD": BLOCK_SIZE_USD,
        "Indicative Market Cost USD": market_cost,
        "Expected Return Proxy": float(total_return_12m[ticker]),
        "Annual Volatility": float(ann_vol[ticker]),
        "Current Price (USD)": current_price,
        "Mean Daily Return": float(mean_daily[ticker]),
        "Std Daily Return": float(std_daily[ticker]),
        "Allowed": 1,
        "Fixed Initial Holding": 0,
        "Variable Candidate": 1,
        "Price Source Status": "Imported via yfinance",
        "Source URL": "https://pypi.org/project/yfinance/",
    })

assets_df = pd.DataFrame(asset_rows)

# -------------------------
# Workbook styling
# -------------------------

wb = Workbook()

dark = PatternFill("solid", fgColor="1F4E78")
section = PatternFill("solid", fgColor="D9EAF7")
light_green = PatternFill("solid", fgColor="E2F0D9")
light_yellow = PatternFill("solid", fgColor="FFF2CC")

white_bold = Font(color="FFFFFF", bold=True)
bold = Font(bold=True)

# -------------------------
# ReadMe
# -------------------------

ws = wb.active
ws.title = "ReadMe"

ws["A1"] = "Workbook overview"
ws["A1"].fill = dark
ws["A1"].font = white_bold

readme_lines = [
    "Demo case: USD 2m Technology & Digital Infrastructure equity sleeve.",
    "The workbook contains 40 asset blocks in total.",
    "10 asset blocks are fixed existing positions and represent USD 1m of existing portfolio capital.",
    "30 asset blocks are variable candidate positions and represent the optimizer decision universe.",
    "The optimizer should select 10 variable blocks of USD 100k each from the 30 candidate blocks.",
    "Only rows with Decision Type = variable should become QUBO / QAOA decision variables.",
    "Rows with Decision Type = fixed should be added back when calculating final portfolio return, variance, holdings and exposure.",
    "Approx Cost USD is deliberately set to USD 100k per block for a clean discrete optimization example.",
    "Indicative Market Cost USD shows the cost implied by rounded share counts and latest imported market price.",
    "If refresh_market_data=1 is used later, ensure the processor does not unintentionally overwrite Approx Cost USD unless this is desired.",
]

for i, line in enumerate(readme_lines, start=3):
    ws.cell(i, 1, line)

ws.column_dimensions["A"].width = 140

# -------------------------
# Assets
# -------------------------

ws = wb.create_sheet("Assets")

ws["A1"] = "Asset block universe"
ws["A1"].fill = dark
ws["A1"].font = white_bold

asset_headers = list(assets_df.columns)

for c, h in enumerate(asset_headers, start=1):
    cell = ws.cell(2, c, h)
    cell.fill = section
    cell.font = bold

for r, row in enumerate(assets_df.itertuples(index=False), start=3):
    for c, value in enumerate(row, start=1):
        ws.cell(r, c, value)

decision_type_col = asset_headers.index("Decision Type") + 1

for r in range(3, ws.max_row + 1):
    decision_type = ws.cell(r, decision_type_col).value
    if decision_type == "fixed":
        for c in range(1, ws.max_column + 1):
            ws.cell(r, c).fill = light_green
    elif decision_type == "variable":
        for c in range(1, ws.max_column + 1):
            ws.cell(r, c).fill = light_yellow

money_cols = ["J", "K", "N"]
for col in money_cols:
    for r in range(3, ws.max_row + 1):
        ws[f"{col}{r}"].number_format = '#,##0.00'

decimal_cols = ["L", "M", "O", "P"]
for col in decimal_cols:
    for r in range(3, ws.max_row + 1):
        ws[f"{col}{r}"].number_format = '0.0000'

asset_widths = {
    "A": 14, "B": 14, "C": 10, "D": 28, "E": 34, "F": 32, "G": 14,
    "H": 14, "I": 10, "J": 16, "K": 22, "L": 20, "M": 18, "N": 18,
    "O": 18, "P": 16, "Q": 10, "R": 18, "S": 18, "T": 24, "U": 30,
}

for col, width in asset_widths.items():
    ws.column_dimensions[col].width = width

ws.freeze_panes = "A3"
ws.auto_filter.ref = ws.dimensions

ws["A2"].comment = Comment("Unique identifier for each asset block. Keep unique per row.", "OpenAI")
ws["B2"].comment = Comment("fixed = existing holding, variable = optimizer decision block.", "OpenAI")
ws["J2"].comment = Comment("The optimizer uses this as the block cost. Set to 100k for this clean demo.", "OpenAI")
ws["K2"].comment = Comment("Informational value based on rounded shares times latest imported price.", "OpenAI")
ws["R2"].comment = Comment("1 for fixed existing holdings, 0 otherwise.", "OpenAI")
ws["S2"].comment = Comment("1 for variable candidate blocks, 0 otherwise.", "OpenAI")

# -------------------------
# Settings
# -------------------------

ws = wb.create_sheet("Settings")

ws["A1"] = "Model and solver settings"
ws["A1"].fill = dark
ws["A1"].font = white_bold

for c, h in enumerate(["Key", "Value", "Description"], start=1):
    cell = ws.cell(2, c, h)
    cell.fill = section
    cell.font = bold

for r, row in enumerate(SETTINGS_ROWS, start=3):
    for c, value in enumerate(row, start=1):
        ws.cell(r, c, value)

ws.column_dimensions["A"].width = 42
ws.column_dimensions["B"].width = 24
ws.column_dimensions["C"].width = 90
ws.freeze_panes = "A3"

# -------------------------
# Returns
# -------------------------

ws = wb.create_sheet("Returns")

ws["A1"] = "Per-ticker return and volatility inputs"
ws["A1"].fill = dark
ws["A1"].font = white_bold

return_headers = [
    "Ticker",
    "Company",
    "Decision Type",
    "12M Return Proxy",
    "Annual Volatility",
    "Mean Daily Return",
    "Std Daily Return",
    "Source",
]

for c, h in enumerate(return_headers, start=1):
    cell = ws.cell(2, c, h)
    cell.fill = section
    cell.font = bold

for r, row in enumerate(assets_df.itertuples(index=False), start=3):
    ticker = row.Ticker
    ws.cell(r, 1, ticker)
    ws.cell(r, 2, row.Company)
    ws.cell(r, 3, row._1 if hasattr(row, "_1") else assets_df.loc[r - 3, "Decision Type"])
    ws.cell(r, 4, float(total_return_12m[ticker]))
    ws.cell(r, 5, float(ann_vol[ticker]))
    ws.cell(r, 6, float(mean_daily[ticker]))
    ws.cell(r, 7, float(std_daily[ticker]))
    ws.cell(r, 8, "https://pypi.org/project/yfinance/")

for col in ["D", "E", "F", "G"]:
    for r in range(3, ws.max_row + 1):
        ws[f"{col}{r}"].number_format = '0.0000'

for col, width in {
    "A": 10, "B": 28, "C": 14, "D": 18, "E": 18, "F": 18, "G": 18, "H": 30
}.items():
    ws.column_dimensions[col].width = width

ws.freeze_panes = "A3"

# -------------------------
# Covariance sheets
# -------------------------

for title, matrix_df, sheet_name in [
    ("Daily return covariance matrix", daily_cov, "Covariance"),
    ("Annualized covariance matrix", annual_cov, "AnnualizedCovariance"),
]:
    ws = wb.create_sheet(sheet_name)
    ws["A1"] = title
    ws["A1"].fill = dark
    ws["A1"].font = white_bold

    tickers = list(matrix_df.index)

    for c, ticker in enumerate(tickers, start=2):
        cell = ws.cell(2, c, ticker)
        cell.fill = section
        cell.font = bold

    for r, ticker in enumerate(tickers, start=3):
        cell = ws.cell(r, 1, ticker)
        cell.fill = section
        cell.font = bold

        for c, ticker2 in enumerate(tickers, start=2):
            ws.cell(r, c, float(matrix_df.loc[ticker, ticker2]))

    ws.freeze_panes = "B3"

    for col_cells in ws.columns:
        letter = col_cells[0].column_letter
        ws.column_dimensions[letter].width = 12

# -------------------------
# Price history
# -------------------------

ws = wb.create_sheet("PriceHistory")

ws["A1"] = "Adjusted close price history"
ws["A1"].fill = dark
ws["A1"].font = white_bold

ws.cell(2, 1, "Date").fill = section
ws.cell(2, 1).font = bold

for c, ticker in enumerate(TICKERS, start=2):
    ws.cell(2, c, ticker).fill = section
    ws.cell(2, c).font = bold

for r, dt in enumerate(prices.index, start=3):
    ws.cell(r, 1, dt.to_pydatetime())
    for c, ticker in enumerate(TICKERS, start=2):
        ws.cell(r, c, float(prices.loc[dt, ticker]))

ws.freeze_panes = "B3"
ws.column_dimensions["A"].width = 16

for c in range(2, len(TICKERS) + 2):
    ws.column_dimensions[ws.cell(2, c).column_letter].width = 12

# -------------------------
# DemoCase
# -------------------------

ws = wb.create_sheet("DemoCase")

ws["A1"] = "Professional demo case"
ws["A1"].fill = dark
ws["A1"].font = white_bold

demo_rows = [
    ("Use case", "Rebalancing a USD 2m Technology & Digital Infrastructure equity sleeve"),
    ("Client context", "The client already holds 10 core positions. These are treated as fixed due to conviction, tax, client preference or transition constraints."),
    ("Optimization question", "Select 10 additional USD 100k blocks from a pre-screened universe of 30 candidates."),
    ("Fixed capital", FIXED_BUDGET_USD),
    ("Variable budget", VARIABLE_BUDGET_USD),
    ("Final target portfolio", "20 positions of approximately USD 100k each"),
    ("Decision variables", "30 variable candidate blocks"),
    ("QAOA interpretation", "Each variable candidate block is one binary decision variable."),
    ("Important processor rule", "Fixed rows are not QUBO variables. They are added back when calculating final portfolio metrics."),
]

for r, row in enumerate(demo_rows, start=3):
    ws.cell(r, 1, row[0])
    ws.cell(r, 2, row[1])

ws.column_dimensions["A"].width = 28
ws.column_dimensions["B"].width = 110

for r in range(3, 3 + len(demo_rows)):
    ws.cell(r, 1).font = bold

# -------------------------
# Sources
# -------------------------

ws = wb.create_sheet("Sources")

ws["A1"] = "Sources"
ws["A1"].fill = dark
ws["A1"].font = white_bold

source_rows = [
    ("Historical market data", "https://pypi.org/project/yfinance/"),
    ("Generated", "This workbook was generated by a Python script using yfinance."),
    ("Demo universe", "Illustrative Technology & Digital Infrastructure equity sleeve for QAOA portfolio optimization."),
]

for r, row in enumerate(source_rows, start=2):
    ws.cell(r, 1, row[0])
    ws.cell(r, 2, row[1])

ws.column_dimensions["A"].width = 28
ws.column_dimensions["B"].width = 110

# -------------------------
# Basic formatting
# -------------------------

for sheet in wb.worksheets:
    for row in sheet.iter_rows():
        for cell in row:
            cell.font = Font(name="Calibri", size=11, bold=cell.font.bold, color=cell.font.color)

    for col_cells in sheet.columns:
        letter = col_cells[0].column_letter
        if sheet.column_dimensions[letter].width is None:
            max_len = max(
                len(str(cell.value)) if cell.value is not None else 0
                for cell in col_cells[: min(len(col_cells), 200)]
            )
            sheet.column_dimensions[letter].width = min(max(max_len + 2, 10), 32)

# Restore title/header fonts after global font pass
for sheet in wb.worksheets:
    if sheet["A1"].value:
        sheet["A1"].font = white_bold
    for cell in sheet[2]:
        if cell.value is not None:
            cell.font = bold

wb.save(OUTPUT_PATH)

print(f"Saved workbook to {OUTPUT_PATH.resolve()}")
print()
print("Generated demo workbook:")
print(f"- Total blocks:      {len(assets_df)}")
print(f"- Fixed blocks:      {(assets_df['Decision Type'] == 'fixed').sum()}")
print(f"- Variable blocks:   {(assets_df['Decision Type'] == 'variable').sum()}")
print(f"- Fixed budget:      USD {FIXED_BUDGET_USD:,.0f}")
print(f"- Variable budget:   USD {VARIABLE_BUDGET_USD:,.0f}")
print(f"- Target new blocks: {N_SELECT_VARIABLE}")
print()
print("Processor reminder:")
print("Use only Decision Type == 'variable' rows as QUBO / QAOA decision variables.")
print("Add fixed rows back when calculating final portfolio metrics.")